In [ ]:
%pip install python-dotenv

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [2]:
from langchain_openai import ChatOpenAI

# temperature=0 -> stable answers, so score changes come from our changes, not randomness
model = ChatOpenAI(
    model="gpt-4.1-mini",
    api_key=OPENAI_API_KEY,
    temperature=0
)

## Document loader

In [3]:
%pip install -U langchain-community pypdf

^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install unstructured[pdf]

In [3]:
from langchain_community.document_loaders import DirectoryLoader

C:\Users\Hello\AppData\Local\Temp\ipykernel_11124\714825054.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader
c:\Users\Hello\Innomatics\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from pathlib import Path

pdf_folder = Path(r"D:\Helpdesk")

print("Folder exists:", pdf_folder.exists())

pdf_files = list(pdf_folder.rglob("*.pdf"))

print("Total PDFs found:", len(pdf_files))

for pdf in pdf_files:
    print(pdf)

Folder exists: True
Total PDFs found: 22
D:\Helpdesk\General Remote Desktop connection troubleshooting - Windows Server _ Microsoft Learn.pdf
D:\Helpdesk\Guidance for Troubleshooting DHCP Authorization Failures - Windows Server _ Microsoft Learn.pdf
D:\Helpdesk\Guidance for troubleshooting DNS - Windows Server _ Microsoft Learn.pdf
D:\Helpdesk\Guidance for troubleshooting printing issues - Windows Server _ Microsoft Learn.pdf
D:\Helpdesk\Guidance for troubleshooting Windows Update issues - Windows Client _ Microsoft Learn.pdf
D:\Helpdesk\Incident Response _ CSRC.pdf
D:\Helpdesk\Install and manage packages - Ubuntu Server documentation.pdf
D:\Helpdesk\Microsoft 365 troubleshooting for admins and IT professionals - Outlook _ Microsoft Learn.pdf
D:\Helpdesk\Microsoft troubleshooting documentation _ Microsoft Learn.pdf
D:\Helpdesk\Networking overview - Windows Client _ Microsoft Learn.pdf
D:\Helpdesk\OWASP API Security Top 10 - OWASP API Security Top 10.pdf
D:\Helpdesk\Printer Not Respondi

In [5]:
from langchain_community.document_loaders import PyPDFLoader

documents = []

for pdf in pdf_files:
    try:
        documents.extend(PyPDFLoader(str(pdf)).load())
        print("Loaded:", pdf.name)
    except Exception as e:
        print("Failed:", pdf.name, "-", e)

print("\nTotal pages loaded:", len(documents))

Loaded: General Remote Desktop connection troubleshooting - Windows Server _ Microsoft Learn.pdf
Loaded: Guidance for Troubleshooting DHCP Authorization Failures - Windows Server _ Microsoft Learn.pdf
Loaded: Guidance for troubleshooting DNS - Windows Server _ Microsoft Learn.pdf
Loaded: Guidance for troubleshooting printing issues - Windows Server _ Microsoft Learn.pdf
Loaded: Guidance for troubleshooting Windows Update issues - Windows Client _ Microsoft Learn.pdf
Loaded: Incident Response _ CSRC.pdf
Loaded: Install and manage packages - Ubuntu Server documentation.pdf
Loaded: Microsoft 365 troubleshooting for admins and IT professionals - Outlook _ Microsoft Learn.pdf
Loaded: Microsoft troubleshooting documentation _ Microsoft Learn.pdf
Loaded: Networking overview - Windows Client _ Microsoft Learn.pdf
Loaded: OWASP API Security Top 10 - OWASP API Security Top 10.pdf
Loaded: Printer Not Responding - Windows Server _ Microsoft Learn.pdf
Loaded: SP 800-61 Rev. 3, Incident Response Rec

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Total document pages:", len(documents))
print("Total chunks created:", len(chunks))
print("\nFirst chunk preview:")
print(chunks[0].page_content[:500] if chunks else "No chunks created")

Total document pages: 119
Total chunks created: 240

First chunk preview:
General Remote Desktop connection
troubleshooting
Applies to: Supported versions of Windows Server
Use these steps when a Remote Desktop client can't connect to a remote desktop but doesn't
provide messages or other symptoms that would help identify the cause.
To check and change the status of the RDP protocol on a local computer, see How to enable
Remote Desktop.
 - It can help you quickly identify and fix common RD Sessions
connectivity issues
Try our Virtual Agent
Check the status of the RDP 


In [7]:
from langchain_openai import OpenAIEmbeddings
embedding =  OpenAIEmbeddings(model ='text-embedding-3-large', api_key = OPENAI_API_KEY,dimensions=1000)

In [8]:
from langchain_chroma import Chroma

In [9]:
vectore_store =  Chroma(embedding_function=embedding,persist_directory='RAG1')

In [10]:
if vectore_store._collection.count() == 0:
    vectore_store.add_documents(chunks)
    print("Chunks added:", vectore_store._collection.count())
else:
    print("Chroma already contains:",
          vectore_store._collection.count(), "chunks")

Chroma already contains: 240 chunks


In [11]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

In [12]:
# k=6 (was 4): gives the reranker a bigger candidate pool, so a higher
# top_n can actually bring in new chunks (helps context recall)
vector_retriever = vectore_store.as_retriever(
    search_kwargs={"k": 6}
)

retriever = BM25Retriever.from_documents(chunks)
retriever.k = 6

ensemble_retriever = EnsembleRetriever(
    retrievers=[retriever, vector_retriever],
    weights=[0.5, 0.5]
)

In [13]:
query = "How do I troubleshoot DNS resolution problems on a Windows computer?"

results = ensemble_retriever.invoke(query)

print("Retrieved documents:", len(results))

for i, doc in enumerate(results, 1):
    print(f"\nResult {i}:")
    print(doc.page_content[:500])
    print("Source:", doc.metadata.get("source"))

Retrieved documents: 11

Result 1:
Troubleshooting DNS servers
Applies to: ✅  Windows Server 2025, ✅  Windows Server 2022, ✅  Windows Server 2019, ✅  Windows Server
2016
 - It can help you quickly identify and fix common DNS issues.
This article discusses how to troubleshoot issues on DNS servers.
1. Run ipconfig /all  at a command prompt, and verify the IP address, subnet mask, and
default gateway.
2. Check whether the DNS server is authoritative for the name that is being looked up. If so,
see Checking for problems with authori
Source: D:\Helpdesk\Troubleshooting DNS Servers _ Microsoft Learn.pdf

Result 2:
Windows Command Prompt
For example:
Windows Command Prompt
If all four of these tests were successful, run ipconfig /displaydns  and check the output for the
name that failed. If you see "Name does not exist" under the failing name, a negative response
was returned from a DNS server and was cached on the client.
To resolve the issue, clear the cache by running ipconfig /flushdns .

### Augementation

In [14]:
from langchain_core.prompts import ChatPromptTemplate


# Balanced prompt:
# - faithfulness: every statement must come from the context (no outside knowledge)
# - relevancy: start directly with steps, no intro / follow-up questions / source names
# - recall: use ALL relevant steps found in the context (not only a few)
prompt = ChatPromptTemplate.from_template("""
You are an IT Helpdesk Assistant.

Answer the user's question using ONLY the provided context.

Rules:
1. Start directly with the answer. Give clear, numbered troubleshooting
   steps that directly address the question.
2. Every step, command, setting, and fact must appear in the context.
   Do not add steps, commands, causes, or advice from your own knowledge,
   even if they are common or correct.
3. Copy commands exactly as written in the context. Do not change or
   add command options.
4. Cover all relevant steps found in the context for this question,
   so the answer is complete but not padded.
5. If the context applies only to a specific product or version,
   say so or leave it out. Do not generalize.
6. Leave out unrelated scenarios and side topics.
7. Do not mention document names, source names, NIST, or any other
   standard in the answer. Sources are shown separately.
8. Do not end with follow-up questions or offers of more help.
9. If the context covers only part of the question, answer only that part.
   If it has nothing relevant, reply exactly:
   "I could not find this in the knowledge base."
10. Keep the answer focused (maximum 8 steps).

Context:
{context}

Question:
{question}

Answer:
""")

In [15]:
query = "What should I check when a printer is not responding?"

docs = ensemble_retriever.invoke(query)

print("Retrieved documents:", len(docs))

for i, doc in enumerate(docs, 1):
    print(f"\n--- Document {i} ---")
    print(doc.page_content[:700])
    print("Source:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page"))

Retrieved documents: 11

--- Document 1 ---
Stopped state, add the Status Reason column to see if there's an error, such as Out of toner or
Off-line.
Right-click the printer icon and select Properties. Select the General tab and check the device
status message. It might indicate a problem or provide an error code to check online or in the
printer manual.
1. Check the printer connection
2. Check if the printer is in a problematic state
3. Check the printer properties
4. Restart the printer
10/27/26, 1:51 PM Printer Not Responding - Windows Server | Microsoft Learn
https://learn.microsoft.com/en-us/troubleshoot/windows-server/printing/printer-not-responding?utm_source=chatgpt.com 1/3
Source: D:\Helpdesk\Printer Not Responding - Windows Server _ Microsoft Learn.pdf
Page: 0

--- Document 2 ---
a result, the drivers may not download.
Workaround: Users with admin privileges can still install printer drivers on the client through
other means, such as copying packaged drivers from a known good

### Reranker

In [ ]:
%pip install sentence-transformers

In [16]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights: 100%|██████████| 105/105 [00:01<00:00, 84.11it/s]


In [ ]:
TOP_N = 4

In [18]:
def rerank_documents(question, documents, top_n=TOP_N):
    pairs = [
        (question, doc.page_content)
        for doc in documents
    ]

    scores = reranker.predict(pairs)

    ranked_docs = sorted(
        zip(documents, scores),
        key=lambda item: item[1],
        reverse=True
    )

    return [
        doc for doc, score in ranked_docs[:top_n]
    ]

In [19]:
query = "How do I troubleshoot DNS resolution problems on a Windows computer?"

retrieved_docs = ensemble_retriever.invoke(query)
reranked_docs = rerank_documents(query, retrieved_docs, top_n=TOP_N)

print("Documents before reranking:", len(retrieved_docs))
print("Documents after reranking:", len(reranked_docs))

for i, doc in enumerate(reranked_docs, 1):
    print(f"\n--- Document {i} ---")
    print("Source:", doc.metadata.get("source", "Unknown"))
    print("Content:", doc.page_content[:300])

Documents before reranking: 11
Documents after reranking: 4

--- Document 1 ---
Source: D:\Helpdesk\Troubleshooting DNS Servers _ Microsoft Learn.pdf
Content: Troubleshooting DNS servers
Applies to: ✅  Windows Server 2025, ✅  Windows Server 2022, ✅  Windows Server 2019, ✅  Windows Server
2016
 - It can help you quickly identify and fix common DNS issues.
This article discusses how to troubleshoot issues on DNS servers.
1. Run ipconfig /all  at a command p

--- Document 2 ---
Source: D:\Helpdesk\Troubleshooting DNS clients _ Microsoft Learn.pdf
Content: Troubleshooting DNS clients
Applies to: ✅  Windows Server 2025, ✅  Windows Server 2022, ✅  Windows Server 2019, ✅  Windows Server
2016
This article discusses how to troubleshoot issues from DNS clients.
1. Open a Command Prompt window as an administrator on the client computer.
2. Run the following 

--- Document 3 ---
Source: D:\Helpdesk\Guidance for troubleshooting DNS - Windows Server _ Microsoft Learn.pdf
Content: If you clear the DN

In [20]:
def answer_from_docs(question, docs):
    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    messages = prompt.invoke({
        "context": context,
        "question": question
    })

    response = model.invoke(messages)
    answer = response.content.strip()

    sources = list(dict.fromkeys(
        doc.metadata.get("source", "Unknown")
        for doc in docs
    ))

    return answer, sources

In [21]:
query = "What should I check when a printer is not responding?"

docs = ensemble_retriever.invoke(query)

reranked_docs = rerank_documents(
    query, docs, top_n=TOP_N
)

answer, sources = answer_from_docs(
    query, reranked_docs
)

print("QUESTION:", query)
print("\nANSWER:", answer)

print("\nSOURCES:")
for doc in reranked_docs:
    print("Source:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page"))

QUESTION: What should I check when a printer is not responding?

ANSWER: 1. Ensure the printer is properly connected to your computer or network:
   - For wired printers, make sure the cable is securely plugged in.
   - For wireless printers, verify that the printer is connected to a functioning Wi-Fi network.

2. Check the printer's display panel or the printer status on your computer. If the printer is in a Stopped state, add the Status Reason column to see if there's an error such as Out of toner or Off-line.

3. Right-click the printer icon and select Properties. Under the General tab, check the device status message for any problems or error codes.

4. Restart the printer.

5. Restart the Print Spooler service:
   - In the Services application, right-click Print Spooler and select Restart.
   - Or, open Command Prompt as administrator and run:
     ```
     net stop spooler
     net start spooler
     ```
   - Or, use Windows PowerShell and run:
     ```
     Restart-Service -Name

In [22]:
def hybrid_rag_answer(question):
    retrieved_docs = ensemble_retriever.invoke(question)
    reranked_docs = rerank_documents(
        question, retrieved_docs, top_n=TOP_N
    )

    answer, sources = answer_from_docs(
        question, reranked_docs
    )

    print("Answer:\n", answer)
    print("\nSources:")
    for source in sources:
        print("-", source)

    return answer

In [23]:
answer = hybrid_rag_answer(
    "How do I troubleshoot DNS resolution problems on a Windows computer?"
)

print(answer)

Answer:
 1. Open a Command Prompt window as an administrator on the client computer.  
2. Run the command `ipconfig /all` and verify the IP address, subnet mask, and default gateway.  
3. Check the DNS servers listed in the output and verify that the IP addresses are correct.  
4. Check the connection-specific DNS suffix in the output and verify that it is correct.  
5. If the client does not have a valid TCP/IP configuration, run `ipconfig /renew` to manually renew the IP address configuration with the DHCP server.  
6. On the DNS server, run `nslookup <name> <IP address of the DNS server>` (for example, `nslookup app1 10.0.0.1`) to test name resolution.  
7. If you get a failure or time-out response, check for recursion problems on the DNS server.  
8. Flush the DNS server cache by running `dnscmd /clearcache` in an administrative Command Prompt window to clear any cached DNS data that might be causing issues.

Sources:
- D:\Helpdesk\Troubleshooting DNS Servers _ Microsoft Learn.pdf


In [24]:
# Out-of-scope check: the system should say it cannot find the answer
answer = hybrid_rag_answer(
    "How do I configure a Kubernetes cluster?"
)

Answer:
 I could not find this in the knowledge base.

Sources:
- D:\Helpdesk\Ubuntu Server documentation.pdf
- D:\Helpdesk\Guidance for troubleshooting DNS - Windows Server _ Microsoft Learn.pdf
- D:\Helpdesk\Troubleshooting DNS Servers _ Microsoft Learn.pdf


### Evalutation

In [27]:
%pip install "langchain-community==0.4.1"


[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
  Attempting uninstall: langchain-community
    Found existing installation: langchain-community 0.4.2
    Uninstalling langchain-community-0.4.2:
      Successfully uninstalled langchain-community-0.4.2
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
import ragas

In [29]:
from ragas import evaluate

In [30]:
# NOTE: verify each reference answer against your PDFs before trusting the scores.
test_questions = [
    "How do I troubleshoot DNS resolution problems on a Windows computer?",
    "What should I check when a printer is not responding?",
    "How can I troubleshoot Remote Desktop connection failures?",
    "How do I troubleshoot a Windows blue screen error?",
    "How can I investigate a cybersecurity incident?",
    "How do I troubleshoot DNS client problems?",
    "How do I troubleshoot problems on a DNS server?",
    "How do I restart the print spooler service?",
    "What causes stop errors in Windows?",
    "What are the common causes of Remote Desktop connection failures?",
    "How do I clear the DNS cache?",
    "What should I verify in the IP configuration when DNS fails on a client?",
    "What should I check when printing fails for all users?",
    "What should I do if a DNS client keeps getting a cached negative response?",
    "What are the phases of incident response?",
]

reference_answers = [
    "Run ipconfig /all to verify IP address, subnet mask, default gateway and DNS servers. Use nslookup against the DNS server, check authoritative data and recursion, and clear the cache.",
    "Check the printer connection (cable or Wi-Fi), check the printer status and stopped state, check the printer properties, restart the printer, restart the print spooler service, and verify the print queue.",
    "Check that Remote Desktop is enabled, check whether Group Policy blocks it, and review session limits, port assignment, authentication and encryption settings, and certificates.",
    "Note the stop code, install the latest Windows updates, update BIOS and firmware, run hardware and memory diagnostics, scan for malware, and check third-party drivers.",
    "Follow the incident response guidance: preparation, detection and analysis, containment, eradication and recovery.",
    "Open an administrator Command Prompt and run ipconfig /all. Verify the IP address, subnet mask, default gateway, DNS server addresses and connection-specific DNS suffix. For dynamic clients use ipconfig /renew.",
    "Run ipconfig /all to verify IP configuration, check whether the server is authoritative for the name, use nslookup against the DNS server, check for recursion problems, and clear the server cache with dnscmd /clearcache.",
    "Restart the print spooler service using net start spooler or the PowerShell command Restart-Service -Name spooler.",
    "Most stop errors are caused by third-party driver code, some by hardware issues, a few by Microsoft code, and the rest have unknown causes because memory is too corrupted to analyze.",
    "Common causes include limits on simultaneous Remote Desktop sessions, port assignment conflicts, incorrect authentication and encryption settings, and corrupted certificates.",
    "Clear the DNS server cache with dnscmd /clearcache from an administrative Command Prompt. On a client, clear the resolver cache with ipconfig /flushdns.",
    "Verify a valid IP address, subnet mask, default gateway, correct DNS server addresses, and the correct connection-specific DNS suffix. Use ipconfig /renew for dynamic clients.",
    "Verify the physical printer is operational, verify the correct printer driver, check the print spooler service is running, check printer connectivity with ping, and check communication with the print server.",
    "Run ipconfig /displaydns and look for Name does not exist for the failing name, then clear the client cache with ipconfig /flushdns.",
    "Preparation, detection and analysis, containment, eradication and recovery, and post-incident activity.",
]

assert len(test_questions) == len(reference_answers)

In [31]:
from datasets import Dataset

def build_eval_dataset(top_n):
    reranked_answers = []
    reranked_contexts = []

    for question in test_questions:
        # 1. Retrieve using BM25 + Chroma
        docs = ensemble_retriever.invoke(question)

        # 2. Rerank the retrieved documents
        docs = rerank_documents(question, docs, top_n=top_n)

        # 3. Store the exact context used for the answer
        reranked_contexts.append(
            [doc.page_content for doc in docs]
        )

        # 4. Generate the answer from that same context (same pipeline as the app)
        answer, sources = answer_from_docs(question, docs)
        reranked_answers.append(answer)

    print("top_n:", top_n)
    print("Questions:", len(test_questions))
    print("Answers:", len(reranked_answers))
    print("Contexts:", len(reranked_contexts))

    dataset = Dataset.from_dict({
        "question": test_questions,
        "answer": reranked_answers,
        "contexts": reranked_contexts,
        "ground_truth": reference_answers
    })

    return dataset, reranked_answers

In [32]:
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

In [33]:
# Evaluate with the chosen TOP_N
eval_dataset, reranked_answers = build_eval_dataset(TOP_N)

result = evaluate(
    dataset=eval_dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ]
)

print(result)

top_n: 4
Questions: 15
Answers: 15
Contexts: 15


Evaluating: 100%|██████████| 60/60 [00:39<00:00,  1.52it/s]


{'faithfulness': 0.8190, 'answer_relevancy': 0.9315, 'context_precision': 0.9000, 'context_recall': 0.9000}


In [35]:
# Per-question scores - shows which questions are still weak
result_df = result.to_pandas()
print(result_df[["user_input", "faithfulness", "answer_relevancy",
                 "context_precision", "context_recall"]]
      if "user_input" in result_df.columns else result_df)

                                           user_input  faithfulness  \
0   How do I troubleshoot DNS resolution problems ...      0.928571   
1   What should I check when a printer is not resp...      1.000000   
2   How can I troubleshoot Remote Desktop connecti...      1.000000   
3   How do I troubleshoot a Windows blue screen er...      1.000000   
4     How can I investigate a cybersecurity incident?      0.000000   
5          How do I troubleshoot DNS client problems?      1.000000   
6     How do I troubleshoot problems on a DNS server?      1.000000   
7         How do I restart the print spooler service?      1.000000   
8                 What causes stop errors in Windows?      1.000000   
9   What are the common causes of Remote Desktop c...      1.000000   
10                      How do I clear the DNS cache?      0.500000   
11  What should I verify in the IP configuration w...      1.000000   
12  What should I check when printing fails for al...      1.000000   
13  Wh

In [36]:
query = "How do I troubleshoot DNS resolution problems on a Windows computer?"

docs = ensemble_retriever.invoke(query)

reranked_docs = rerank_documents(
    query, docs, top_n=TOP_N
)

answer, sources = answer_from_docs(
    query, reranked_docs
)

print("QUESTION:", query)
print("\nANSWER:", answer)

print("\nSOURCES:")
for doc in reranked_docs:
    print("Source:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page"))

QUESTION: How do I troubleshoot DNS resolution problems on a Windows computer?

ANSWER: 1. Open a Command Prompt window as an administrator on the client computer.  
2. Run the command `ipconfig /all` and verify the IP address, subnet mask, and default gateway.  
3. Check the DNS servers listed in the output and verify that the IP addresses are correct.  
4. Check the connection-specific DNS suffix in the output and verify that it is correct.  
5. If the client does not have a valid TCP/IP configuration, run `ipconfig /renew` to manually renew the IP address configuration with the DHCP server.  
6. On the DNS server, run `nslookup <name> <IP address of the DNS server>` (for example, `nslookup app1 10.0.0.1`) to test name resolution.  
7. If you get a failure or time-out response, investigate recursion problems on the DNS server.  
8. Flush the DNS resolver cache on the DNS server by running `dnscmd /clearcache` in an administrative Command Prompt window.

SOURCES:
Source: D:\Helpdesk\T

In [37]:
query = "How do I troubleshoot DNS resolution problems on a Windows computer?"

retrieved_docs = ensemble_retriever.invoke(query)
reranked_docs = rerank_documents(query, retrieved_docs, top_n=TOP_N)

for i, doc in enumerate(reranked_docs, 1):
    print(f"\n--- Document {i} ---")
    print("Source:", doc.metadata.get("source", "Unknown"))
    print("Page:", doc.metadata.get("page", "Unknown"))
    print(doc.page_content[:1000])


--- Document 1 ---
Source: D:\Helpdesk\Troubleshooting DNS Servers _ Microsoft Learn.pdf
Page: 0
Troubleshooting DNS servers
Applies to: ✅  Windows Server 2025, ✅  Windows Server 2022, ✅  Windows Server 2019, ✅  Windows Server
2016
 - It can help you quickly identify and fix common DNS issues.
This article discusses how to troubleshoot issues on DNS servers.
1. Run ipconfig /all  at a command prompt, and verify the IP address, subnet mask, and
default gateway.
2. Check whether the DNS server is authoritative for the name that is being looked up. If so,
see Checking for problems with authoritative data.
3. Run the following command:
Windows Command Prompt
For example:
Windows Command Prompt
If you get a failure or time-out response, see Checking for recursion problems.
4. Flush the resolver cache. To do this, run the following command in an administrative
Command Prompt window:
Windows Command Prompt
Try our Virtual Agent
Check IP configuration
nslookup <name> <IP address of the DNS se

In [38]:
for i, doc in enumerate(reranked_docs, 1):
    print(f"\nDocument {i}")
    print("Source:", doc.metadata.get("source", "Unknown"))
    print("Page:", doc.metadata.get("page", "Unknown"))
    print("Content:", doc.page_content[:300])


Document 1
Source: D:\Helpdesk\Troubleshooting DNS Servers _ Microsoft Learn.pdf
Page: 0
Content: Troubleshooting DNS servers
Applies to: ✅  Windows Server 2025, ✅  Windows Server 2022, ✅  Windows Server 2019, ✅  Windows Server
2016
 - It can help you quickly identify and fix common DNS issues.
This article discusses how to troubleshoot issues on DNS servers.
1. Run ipconfig /all  at a command p

Document 2
Source: D:\Helpdesk\Troubleshooting DNS clients _ Microsoft Learn.pdf
Page: 0
Content: Troubleshooting DNS clients
Applies to: ✅  Windows Server 2025, ✅  Windows Server 2022, ✅  Windows Server 2019, ✅  Windows Server
2016
This article discusses how to troubleshoot issues from DNS clients.
1. Open a Command Prompt window as an administrator on the client computer.
2. Run the following 

Document 3
Source: D:\Helpdesk\Guidance for troubleshooting DNS - Windows Server _ Microsoft Learn.pdf
Page: 5
Content: If you clear the DNS server cache on the forwarding computer (the local DNS ser

In [39]:
for i, question in enumerate(test_questions):
    print("\n" + "=" * 60)
    print("QUESTION:", question)
    print("\nREFERENCE:", reference_answers[i])
    print("\nGENERATED:", reranked_answers[i])


QUESTION: How do I troubleshoot DNS resolution problems on a Windows computer?

REFERENCE: Run ipconfig /all to verify IP address, subnet mask, default gateway and DNS servers. Use nslookup against the DNS server, check authoritative data and recursion, and clear the cache.

GENERATED: 1. Open a Command Prompt window as an administrator on the client computer.  
2. Run the command `ipconfig /all` and verify the IP address, subnet mask, and default gateway.  
3. Check the DNS servers listed in the output and verify that the IP addresses are correct.  
4. Check the connection-specific DNS suffix in the output and verify that it is correct.  
5. If the client does not have a valid TCP/IP configuration, run `ipconfig /renew` to manually renew the IP address configuration with the DHCP server.  
6. On the DNS server, run `nslookup <name> <IP address of the DNS server>` (for example, `nslookup app1 10.0.0.1`) to test name resolution.  
7. If you get a failure or time-out response, check for

In [40]:
query = "How do I troubleshoot DNS resolution problems on a Windows computer?"

retrieved_docs = ensemble_retriever.invoke(query)
reranked_docs = rerank_documents(
    query, retrieved_docs, top_n=TOP_N
)

answer, sources = answer_from_docs(query, reranked_docs)

print("ANSWER:\n", answer)
print("\nACTUAL RETRIEVED SOURCES:")
for source in sources:
    print("-", source)

ANSWER:
 1. Open a Command Prompt window as an administrator on the client computer.  
2. Run the command `ipconfig /all` and verify the IP address, subnet mask, and default gateway.  
3. Check the DNS servers listed in the output and verify that the IP addresses are correct.  
4. Check the connection-specific DNS suffix in the output and verify that it is correct.  
5. If the client does not have a valid TCP/IP configuration, run `ipconfig /renew` to manually renew the IP address configuration with the DHCP server.  
6. On the DNS server, run `nslookup <name> <IP address of the DNS server>` (for example, `nslookup app1 10.0.0.1`) to test name resolution.  
7. If you get a failure or time-out response, check for recursion problems on the DNS server.  
8. Flush the DNS server cache by running `dnscmd /clearcache` in an administrative Command Prompt window to clear stale DNS data.

ACTUAL RETRIEVED SOURCES:
- D:\Helpdesk\Troubleshooting DNS Servers _ Microsoft Learn.pdf
- D:\Helpdesk\Tro